In [105]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import text, create_engine, bindparam, Engine
import os
from dotenv import load_dotenv
import re
from datetime import datetime
load_dotenv()
DB_HOST = os.getenv("DB_HOST")
engine = create_engine(DB_HOST)
SUBSTATION_ID = ["CIPC0001"]
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_columns = None
import calendar
BILLING_PERIOD = "202501"

# DATA PROCESSING AND TEMPLATE GENERATOR CLASS

In [246]:
class DSAS_TEMPLATE_GENERATOR:
    def __init__(self, substation_id, billing_period, db_engine:Engine):
        self.substation_id = substation_id
        self.billing_period = billing_period
        self.year = billing_period[:4]
        self.db_engine = db_engine
        
    # READ AND PROCESS DATABASE DATA
    # CONSUMER METER
    def read_and_process_consumer_meter(self):
            # QUERY CONSUMER METER DATA FROM DATA BASE
            stmt = text("""
                        select cm.*, b.substation_id, b.nominal_voltage_kv, sd.phasing from gis.consumer_meter as cm
                        join gis.service_drop as sd
                        on cm.account_no = sd.to_customer_id
                        join gis.bus as b
                        on b.bus_id = sd.from_bus_id
                        where b.substation_id IN :substation
                        """).bindparams(
                bindparam("substation", expanding=True)
            )
            cm_df = gpd.read_postgis(stmt, engine, geom_col='geom', params={"substation": self.substation_id})
    
            # LUMP DATA
            lump_cm_df = cm_df[cm_df['account_type'] == 'Consumer Lump']
            lump_cm_df.sort_values(by='account_no', inplace=True)
            
            
            
            kwh_df = pd.read_csv(f"converted_kwr/{self.year}/{self.billing_period}.csv")
            kwh_df['account_no'] = kwh_df['account_no'].str.replace("-", "", regex=False).str.strip().astype("string")
            kwh_df.kwh_used = kwh_df.kwh_used.apply(lambda x: x.replace(",", "")).astype("float")
            kwh_df.previous_reading = kwh_df.previous_reading.apply(lambda x: x.replace(",", "")).astype("float")
            kwh_df.present_reading = kwh_df.present_reading.apply(lambda x: x.replace(",", "")).astype("float")
            # KWH DATA WITH GEOMETRY
            kwh_df_with_geom = kwh_df.merge(cm_df, how="inner", left_on="account_no", right_on="account_no")
            kwh_df_with_geom = kwh_df_with_geom[['account_no', 'account_name', 'kwh_used', 'account_type', 'substation_id', 'nominal_voltage_kv', 'phasing' ]]
            kwh_df_with_geom.rename(columns={
                'account_no' : 'customer_id',
                'account_name': 'customer_name',
                'account_type' : 'customer_type',   
            },inplace=True)
            
            kwh_df_with_geom['customer_id'].apply(lambda x: x[:6] + "0000").unique()
                    # KWH LUMP DATA
            kwh_df_without_geom = kwh_df.loc[~kwh_df['account_no'].isin(kwh_df_with_geom['customer_id'])]
            kwh_df_without_geom['lump_account_no'] = kwh_df_without_geom['account_no'].apply(lambda x: x[:6] + "0000")
            lump_kwh_filter = lump_cm_df.merge(kwh_df_without_geom, how="inner", left_on="account_no", right_on="lump_account_no")
            lump_kwh_filter = lump_kwh_filter.groupby(["lump_account_no", "account_name", "account_type",'substation_id', 'nominal_voltage_kv', 'phasing']).agg({
                "kwh_used" : "sum"
            }).reset_index()
            
            
            lump_kwh_filter.rename(columns={
                'lump_account_no' : 'customer_id',
                'account_name' : 'customer_name',
                'account_type' : 'customer_type'
            },inplace=True)
            clean_kwh_df = pd.concat([kwh_df_with_geom, lump_kwh_filter])
            
            
    
            
            # MFSR DATA
            mfsr_data = pd.read_csv(f"mfsr/cleaned/mfsr_{self.year}.csv")
            mfsr_data = mfsr_data[['substation_id','Sales in KWH', 'billing_period']]
            mfsr_data = mfsr_data.loc[mfsr_data['billing_period'].astype(str) == str(self.billing_period)]
            mfsr_data = mfsr_data.loc[mfsr_data['substation_id'].astype(str).isin(self.substation_id)]
            
            
            
            # SCALING KWH DATA
            scaled_kwh_df = clean_kwh_df.copy()
            kwh_diff = mfsr_data.copy()
            kwh_sales = mfsr_data['Sales in KWH'].sum()
            kwh_used = scaled_kwh_df['kwh_used'].sum()
            kwh_diff['difference'] = kwh_sales / kwh_used
            kwh_diff = kwh_diff[['substation_id', 'difference', 'billing_period']]
            
            
            scaled_kwh_df = scaled_kwh_df.merge(kwh_diff, how="left", left_on="substation_id", right_on="substation_id")
            scaled_kwh_df[
                'scaled_kwh_used'] = scaled_kwh_df['kwh_used'] * scaled_kwh_df['difference']
            scaled_kwh_df[
                'kwh_used'
            ] = scaled_kwh_df['scaled_kwh_used']
            scaled_kwh_df.drop(columns=['difference', 'scaled_kwh_used'], inplace=True)
            return scaled_kwh_df
    # BUS
    def process_bus_data(self):
        stmt = text("""
            select * from gis.bus
            where substation_id in :substation
            """).bindparams(bindparam("substation", expanding=True))
        bus_df = gpd.read_postgis(stmt, self.db_engine, geom_col='geom', params={"substation": self.substation_id})
        bus_df = bus_df[['bus_id','description','nominal_voltage_kv', 'geom']]
        return bus_df
    # PRIMARY LINE OVERHEAD
    def process_primary_lines(self):
        stmt = text('''
            select pl.*, b.feeder_id,
            cw.type as conductor_type,
            cw.size as conductor_size,
            cw.unit as conductor_unit,
            cw.strand as conductor_strand,
            nc.type as neutral_type,
            nc.size as neutral_size,
            nc.unit as neutral_unit
            from gis.primary_lines as pl
            left join gis.bus as b
            on b.bus_id = pl.from_bus_id
            left join gis.conductor_wires as cw
            on cw.id = pl.conductor_wire
            left join gis.neutral_concentric_cable as nc
            on pl.neutral_wire = nc.id
            where b.substation_id in :substation
            ''').bindparams(bindparam("substation", expanding=True))
        
        pl_df = gpd.read_postgis(stmt, engine, geom_col="geom", params={
            "substation": SUBSTATION_ID})
        pl_spacing = pd.read_excel(r"sample-data\cor_bus_202401.xls", sheet_name="Primary Line-Overhead", engine="calamine")
        spacing_data = pl_spacing.iloc[:, list(range(4, 6)) + list(range(16, pl_spacing.shape[1]))].columns
        pl_spacing = pl_spacing[spacing_data]
        pl_spacing.drop_duplicates(subset=['Phasing', 'Configuration'], inplace=True)
        pl_spacing.columns = pl_spacing.columns.str.replace(r"\(|\)", "", regex=True)
        pl_spacing.columns = pl_spacing.columns.str.replace(r"\s+|-+", "_", regex=True).str.lower()
        pl_spacing['configuration'] = pl_spacing['configuration'].str.upper()
        pl_df = pl_df.merge(pl_spacing, how="left", left_on=["phasing","configuration"], right_on=["phasing","configuration"])
        return pl_df
    # DISTRIBUTION TRANSFORMER
    def process_distribution_transformer(self):
        stmt = text(
            """
            select dt.*, tt.kva_rating, tt.primary_voltage_rating, secondary_voltage_rating from gis.distribution_transformer as dt 
            left join gis.bus as b
            on  b.bus_id = dt.from_primary_bus_id
            left join gis.transformer_type as tt 
            on tt.id = dt.transformer_type
            where b.substation_id in :substation
            
            """
        ).bindparams(bindparam("substation", expanding=True))
        dt_df = gpd.read_postgis(stmt, engine, geom_col='geom', params={"substation": SUBSTATION_ID})


        cor_bus_dt_df = pd.read_excel("./sample-data/cor_bus_202401.xls", sheet_name="Distribution Transformer", engine="calamine")
        unique_values_df = cor_bus_dt_df[
                [
                    'KVA Rating',
                    'Secondary Voltage Rating (kV)',
                    'Primary Voltage Rating(kV)',
                    'X/R Ratio',
                    'No-Load Loss (kW)',
                    'Exciting Current (%)'
                ]
            ]

        unique_values_df = unique_values_df.drop_duplicates()
        dt_df['kva_rating'] = dt_df['kva_rating'].apply(lambda x: x.replace("kVA", "")).astype("float")
        dt_merge = dt_df.merge(unique_values_df, how="left", 
                    left_on=["kva_rating", 'primary_voltage_rating', 'secondary_voltage_rating'], 
                    right_on=["KVA Rating", 'Primary Voltage Rating(kV)', 'Secondary Voltage Rating (kV)'])
        dt_merge.drop(columns=['Primary Voltage Rating(kV)', 'Secondary Voltage Rating (kV)','remarks','image','is_active'], inplace=True)
        dt_clean = dt_merge
        return dt_clean

<div class="alert alert-success">
 <h3>
 <strong>
CONSUMER METER
 </strong>
 </h3>
</div>

In [ ]:
class ConsumerMeter(DSAS_TEMPLATE_GENERATOR):
    def __init__(self, substation_id, billing_period, db_engine:Engine):
        super().__init__(substation_id, billing_period, db_engine)

    def type_classification(self, customer_type, kwh_used):
        """
            Classify customer type
        Args:
            customer_type (_type_): billing consumer type
            kwh_used (_type_): consumer kwh

        Returns:
            _type_: literal[RES1, RES2, RES3, COM1, COM2, COM3]
        """
        if re.match(r"(CIPC|NPC)\s*R" , customer_type, re.IGNORECASE):
            if kwh_used <= 50:
                return "RES1" 
            elif kwh_used <= 150:
                return "RES2"
            elif kwh_used <= 300:
                return "RES3"
            elif kwh_used > 300:
                return "RES4"
        elif re.match(r"(NPC|CIPC)\s+(SC|LC)" , customer_type, re.IGNORECASE):
            if kwh_used <= 300:
                return "COM1"
            elif kwh_used <= 800:
                return "COM2"
            elif kwh_used <= 2500:
                return "COM3"
            elif kwh_used > 2500:
                return "COM4"
        elif re.match(r"(NPC|CIPC)\s+PB" , customer_type, re.IGNORECASE):
            if kwh_used <= 800:
                return "PB1"
            elif kwh_used > 800:
                return "PB2"
        elif re.match(r"(NPC|CIPC)\s+SL" , customer_type, re.IGNORECASE):
            return "SL"
        elif re.match(r"(NPC|CIPC)\s+(I|O)" , customer_type, re.IGNORECASE):
            return "IND"
        elif re.match(r"Consumer\s*Lump" , customer_type, re.IGNORECASE):
            return "RES2"
        else:
            return customer_type
    
    def phase_classification(self, phasing):
        if len(phasing) == 2:
            return 1
        elif len(phasing) == 3:
            return 2
        elif len(phasing) == 4:
            return 3
        else:
            return phasing
    
    # CUSTOMER DATA
    def save_customer_data(self):
        consumer = self.read_and_process_consumer_meter()
        consumer['customer_type'] = consumer.apply(lambda x: self.type_classification(x['customer_type'], x['kwh_used']), axis=1)
        consumer['service_voltage'] = (consumer['nominal_voltage_kv'] * 1000).astype("int")
        consumer['phase'] = consumer.apply(lambda x: self.phase_classification(x['phasing']), axis=1)
        consumer = consumer[['customer_id', 'customer_name', 'service_voltage', 'phase']]
        consumer.insert(0,'count', consumer.index + 1)
        return consumer
    
    
    # BILLING CYCLE
    def calculate_number_of_days(self):
        '''USING CALENDAR MODULE TO DETERMINE NUMBER OF DAYS IN MONTH'''
        period = datetime.strptime(self.billing_period, "%Y%m")
        number_of_days = calendar.monthrange(period.year, period.month)[1]
        return number_of_days
    
    def calculate_number_of_hours(self):
        '''USING CALENDAR MODULE TO DETERMINE NUMBER OF HOURS IN MONTH'''
        period = datetime.strptime(self.billing_period, "%Y%m")
        number_of_hours = calendar.monthrange(period.year, period.month)[1] * 24
        return number_of_hours
    
    def save_billing_cycle(self):
        '''save billing cycle template'''
        billing_cycle = {
            "count": 1, 
            "period": self.billing_period,
            "period_covered": datetime.strptime(self.billing_period, "%Y%m").strftime("%B %Y"),
            "number_of_days": self.calculate_number_of_days(),
            "number_of_hours": self.calculate_number_of_hours()
        }
        billing_cycle_df = pd.DataFrame(billing_cycle, index=[0])
        return billing_cycle_df
    # ---------------------------------------------------------------------------------------------
    # CUSTOMER ENERGY CONSUMPTION
    def status_classification(self, kwh_used):
        '''CLASSIFY CONSUMER STATUS based on the kwh used'''
        if kwh_used == 0:
            return 0
        else:
            return 1
    def save_customer_energy_consumption(self):
        consumer = self.read_and_process_consumer_meter()
        consumer['status'] = consumer['kwh_used'].apply(lambda x: self.status_classification(x))
        consumer['power_factor'] = 0.93
        consumer = consumer[['customer_id', 'billing_period', 'kwh_used', 'power_factor', 'status']]
        consumer.insert(0,'count', consumer.index + 1)
        return consumer
    
    
    def classify_load_curve(self): 
        if re.match(r'CIPC0001|NPCC0001',self.substation_id[0], re.IGNORECASE):
            return 'MAIN LAND'
        else:
            return 'ISLANDS'
        
    # LOAD CURVE DATA
    def save_load_curve_data(self):
        lcdf = pd.read_csv(r"surveyed-load-curve\load_curve.csv")
        used_load_curve = self.classify_load_curve()
        lcdf = lcdf.loc[lcdf['area'] == used_load_curve]
        lcdf.sort_values(by="customer_type", inplace=True)
        provider = re.search(r'[A-Za-z]*',self.substation_id[0]).group(0)
        lcdf.insert(1, 'load_curve_id', provider+lcdf['customer_type'])
        lcdf.drop(columns = ['id','area','max'], inplace=True)
        return lcdf
    


<div class="alert alert-success">
 <h3>
 <strong>
 BUS CLASS
 </strong>
 </h3>
</div>

In [ ]:
class BusData(DSAS_TEMPLATE_GENERATOR):
    def __init__(self, substation_id, billing_period, db_engine:Engine):
        super().__init__(substation_id, billing_period, db_engine)
    
    def save_bus_data(self):
        bus = self.process_bus_data()
        bus.insert(0,'count', bus.index + 1)
        bus.drop(columns=['geom'], inplace=True)    
        return bus

<div class="alert alert-success">
 <h3>
 <strong>
    PRIMARYLINES
 </strong>
 </h3>
</div>

In [239]:
class PrimaryLines(DSAS_TEMPLATE_GENERATOR):
    def __init__(self, substation_id, billing_period, db_engine: Engine):
        super().__init__(substation_id, billing_period, db_engine)

    def save_primary_line_overhead(self):
        primary_lines = self.process_primary_lines()
        primary_lines.insert(0, 'count', primary_lines.index + 1)
        primary_line_overhead = primary_lines[primary_lines['description']
                                              == 'PRIMARY LINE OVERHEAD']
        primary_line_overhead = primary_line_overhead[
            ['count',  'primary_line_id', 'from_bus_id', 'to_bus_id',
             'phasing', 'configuration', 'system_ground_type',
             'length_meters', 'conductor_type',
             'conductor_size', 'conductor_unit', 'conductor_strand', 'neutral_type',
             'neutral_size', 'neutral_unit', 'spacing_d12_meters',
             'spacing_d23_meters', 'spacing_d13_meters', 'spacing_d1n_meters',
             'spacing_d2n_meters', 'spacing_d3n_meters', 'spacing_dc1_c2_meters',
             'height_h1_meters', 'height_h2_meters', 'height_h3_meters',
             'height_hn_meters', 'earth_resistivity_ohm_meter']]
        return primary_line_overhead

    def save_primary_line_underground(self):
        primary_lines = self.process_primary_lines()
        primary_lines.insert(0, 'count', primary_lines.index + 1)
        primary_line_underground = primary_lines[primary_lines['description']
                                                 == 'PRIMARY LINE UNDERGROUND']
        
        primary_line_underground = primary_line_underground[[
            'count',  'primary_line_id', 'from_bus_id', 'to_bus_id',
            'phasing', 'configuration', 'system_ground_type',
            'length_meters', 'conductor_type',
            'conductor_size', 'conductor_unit']]
        primary_line_underground['no_of_cores'] = 1
        primary_line_underground['diameter_underarmour'] = 43.82
        primary_line_underground['armour_wire_diameter']= 0.10
        primary_line_underground['overall_diameter'] = 49.60
        primary_line_underground['ac_resistance_ohm_per_meter'] = 0.09
        primary_line_underground['inductive_resistance'] = 0.25
        primary_line_underground['capacitant'] = 0.22
        primary_line_underground['earth_resistivity_ohm_meter'] = 100
        return primary_line_underground

    def save_pred_rel_data(self):
        primary_lines = self.process_primary_lines()
        pred_rel = primary_lines[['primary_line_id']]
        pred_rel['permanent_failure_rate'] = 0.3
        pred_rel['temporary_failure_rate'] = 0.3
        pred_rel['repair_rate_hour'] = 0.5
        pred_rel.insert(0, 'count', pred_rel.index + 1)
        return pred_rel

<div class="alert alert-success">
 <h3>
 <strong>
    DISTRIBUTION TRANSFORMER
 </strong>
 </h3>
</div>

In [ ]:
class DistributionTransformer(DSAS_TEMPLATE_GENERATOR):
    def __init__(self, substation_id, billing_period, db_engine:Engine):
        super().__init__(substation_id, billing_period, db_engine)
        
    def save_distribution_transformer_data(self):
        distribution_transformer = self.process_distribution_transformer()
        distribution_transformer = distribution_transformer[['transformer_id', 'from_primary_bus_id', 'to_secondary_bus_id','primary_phasing', 'secondary_phasing', 'installation_type', '']]
        distribution_transformer.insert(0, 'count', distribution_transformer.index + 1)
        return distribution_transformer

In [248]:
consumer_meter = ConsumerMeter(SUBSTATION_ID, BILLING_PERIOD, engine)
bus = BusData(SUBSTATION_ID, BILLING_PERIOD, engine)
primary_lines = PrimaryLines(SUBSTATION_ID, BILLING_PERIOD, engine)
primary_lines.save_pred_rel_data()
distribution_transformer = DistributionTransformer(SUBSTATION_ID, BILLING_PERIOD, engine)
distribution_transformer.save_distribution_transformer_data()

,count,id,geom,transformer_id,from_primary_bus_id,primary_phasing,to_secondary_bus_id,secondary_phasing,description,installation_type,connection_code,transformer_type,village_id,municipality_id,kva_rating,primary_voltage_rating,secondary_voltage_rating,KVA Rating,X/R Ratio,No-Load Loss (kW),Exciting Current (%)
0,1,550,POINT (120.20444 11.99826),STA0139,F1_P001A1001,ABCN,OSTA0139,ABCN,Sole,PAD-MOUNTED,4,22,5,1,15.00,13.80,0.24,15.00,1.50,0.01,0.05
1,2,632,POINT (120.1574 12.04086),DTA0001,F4,BN,S000A0031,BN,Distribution Transformer,POLE-MOUNTED,1,12,30,1,10.00,7.96,0.24,10.00,1.50,0.01,0.06
2,3,261,POINT (120.2034 11.99691),STA0125,F2_P002A0054,ABCN,OSTA0125,ABCN,Sole,PAD-MOUNTED,4,10,6,1,100.00,13.80,0.24,100.00,1.50,0.05,0.36
3,4,10,POINT (120.20861 11.99984),STA0449,F1_P001A0424,ABCN,OSTA0449,ABCN,Sole,POLE-MOUNTED,10,25,4,1,167.00,13.80,0.24,167.00,1.50,0.07,0.61
4,5,200,POINT (120.20628 12.00027),STA0194,F1_P001A0125,CN,OSTA0194,CN,Sole,POLE-MOUNTED,1,18,4,1,25.00,7.96,0.24,25.00,1.50,0.02,0.16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,513,353,POINT (120.01277 12.23746),DTB0093,F3_P003B1063,AN,S003B1063,AN,Distribution Transformer,POLE-MOUNTED,1,24,22,2,15.00,7.96,0.24,15.00,1.50,0.01,0.09
513,514,357,POINT (120.01262 12.2436),DTB0096,F3_P003B1101,AN,S003B1101,AN,Distribution Transformer,POLE-MOUNTED,1,12,22,2,10.00,7.96,0.24,10.00,1.50,0.01,0.06
514,515,355,POINT (120.00536 12.23861),DTB0094,F3_P003B1072,AN,S003B1072,AN,Distribution Transformer,POLE-MOUNTED,1,12,22,2,10.00,7.96,0.24,10.00,1.50,0.01,0.06
515,516,356,POINT (120.01145 12.25137),DTB0089,F3_P003B1095,AN,S003B1095,AN,Distribution Transformer,POLE-MOUNTED,1,18,22,2,25.00,7.96,0.24,25.00,1.50,0.02,0.16


#### CHECKING AND VALIDATION ON MONTHLY FINANCIAL SYSTEMLOSS REPORT (MFSR)

Uppon checking on the result of total consumer kwh_used for Coron and Busuanga Sometime the MFSR data is equal to the kwh_used total per consumer. So to scale down or up the kwh_used to match the mfsr data the solution are:
- Compare the different per each municipality mfsr kwh_sale vs per consumer total_kwh
- Then I merge the result different back to clean_kwh_sale and multiply the different into original kwh_sale per consumer to match the total kwh_sale in the mfsr.

<div class="alert alert-info">
<strong>Note:</strong> MFSR data is private.
</div>

<div class="alert alert-success">
<strong> Processing MFSR Data</strong>
</div>



In [138]:

YEAR=2025
mfsr_dir = "mfsr"
class MFSR:
    def __init__(self, folder, YEAR, db_engine:Engine):
        self.folder = folder
        self.clean_df = []
        self.YEAR = YEAR
        self.engine = db_engine
    # READ AND PROCESS
    def read_and_process(self):
        mfsr_file = os.listdir(self.folder)
        for file in [f for f in mfsr_file if re.search(str(self.YEAR), f)]:
            mfsr_sheet = 'Sched 1-GS'
            use_cols = 'b:N'
            skip_row = 9
            # FORMATING BILLING PERIOD
            ptime_format = "%B%Y"
            ftime_format = "%Y%m"
            billing_period = datetime.strptime(file.split("@")[1].split(".")[0], ptime_format).strftime(ftime_format)
            
            # READ AND PROCESS THE DATA
            mfsr_df = pd.read_excel(os.path.join(mfsr_dir, file), sheet_name=mfsr_sheet, skiprows=skip_row, usecols=use_cols)
            mfsr_df.rename(columns={
                mfsr_df.columns[0]: "CATEGORY",
            }, inplace=True)
        
            mfsr_df['CATEGORY'] = mfsr_df['CATEGORY'].str.replace(
                r"^\s*(?:\(\d+\)\s*[A-Za-z]?|[A-Za-z0-9]+\.)\s*", 
                "", 
                regex=True,
            )
            mfsr_df.dropna(subset=mfsr_df.columns[1:], inplace=True, how="all")
            mfsr_df.fillna(0, inplace=True)
            mfsr_df = mfsr_df.iloc[:5, :]
            for mfsr_col in mfsr_df.columns[1:]:
                mfsr_df[mfsr_col] = mfsr_df[mfsr_col].astype(float)
            mfsr_df['CORON_BUSUANGA'] = mfsr_df['CORON'] + mfsr_df['BUSUANGA']
            mfsr_df = mfsr_df.set_index("CATEGORY").T
            mfsr_df.columns.name = None
            mfsr_df.index.name = "FEEDER"
            mfsr_df = mfsr_df.reset_index()
            mfsr_df['billing_period'] = billing_period
            mfsr_df['billing_period'] = mfsr_df['billing_period'].astype("string")
            self.clean_df.append(mfsr_df)
        return self.clean_df
    # SAVE DATA
    def save_data(self,destination:str):
        try: 
            stmt = text("""
                        select sub.substation_id, mun.name as municipality, vill.name as village from gis.substation as sub
                        join gis.villages  as vill
                        on vill.id = sub.village_id
                        join gis.municipality as mun
                        on mun.id = sub.municipal_id;
                        """)
            data = pd.read_sql(stmt, self.engine)
            data = data[data['substation_id'] !='CIPC0002']
            data['FEEDER'] = data['village'].apply(lambda x: 'CORON_BUSUANGA' if re.search("GUADALUPE", x) else 'CULION' if x == 'OSMEÑA' else x)
            mfsr_df = pd.concat(self.clean_df)
            mfsr_df = mfsr_df.merge(data, how="left", left_on="FEEDER", right_on="FEEDER")
            mfsr_df = mfsr_df[~mfsr_df['FEEDER'].isin(['CORON', 'BUSUANGA'])]
            mfsr_df.to_csv(os.path.join(destination, f"mfsr_{self.YEAR}.csv"), index=False)
        except Exception as e:
            raise e
        return f"{destination} saved"

### SAVE THE PROCESS AND CLEAN MFSR DATA

In [139]:
# MFSR DATA
mfsr = MFSR(mfsr_dir, YEAR, engine)
mfsr.read_and_process()
destination_folder = os.path.join(mfsr_dir,"cleaned")
os.makedirs(destination_folder, exist_ok=True)
print(mfsr.save_data(destination_folder))

mfsr\cleaned saved


,phasing,configuration,spacing_d12_meters,spacing_d23_meters,spacing_d13_meters,spacing_d1n_meters,spacing_d2n_meters,spacing_d3n_meters,spacing_dc1_c2_meters,height_h1_meters,height_h2_meters,height_h3_meters,height_hn_meters,earth_resistivity_ohm_meter
0,ABCN,HORIZONTAL,1.07,1.07,2.13,1.62,1.22,1.62,0,11.66,11.66,11.66,10.44,100
1,ABCN,VERTICAL,1.22,1.22,2.44,3.66,2.44,1.22,0,11.66,10.44,9.22,8.00,100
2,ABCN,TRIANGULAR,1.21,1.21,2.24,1.27,1.07,1.27,0,11.43,11.89,11.43,10.82,100
51,AN,VERTICAL,0.00,0.00,0.00,1.07,0.00,0.00,0,11.89,0.00,0.00,10.82,100
98,CN,VERTICAL,0.00,0.00,0.00,1.07,0.00,0.00,0,11.89,0.00,0.00,10.82,100
102,BN,VERTICAL,0.00,0.00,0.00,1.07,0.00,0.00,0,11.89,0.00,0.00,10.82,100
413,BN,HORIZONTAL,0.00,0.00,0.00,1.07,0.00,0.00,0,11.89,0.00,0.00,10.82,100
538,CN,HORIZONTAL,0.00,0.00,0.00,1.07,0.00,0.00,0,11.89,0.00,0.00,10.82,100
